In [17]:
import paarti.utils.maos_utils as mu
import paarti.utils.koa_utils as ku
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from pathlib import Path
from astropy.io import fits

In [ ]:
# Grab KOA image for which to run a MAOS sim
koa_file = Path("/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits")
old_hdr, clean_img = ku.clean_koa(koa_file)

In [18]:
# Compute metrics on cleaned KOA image 
# re-defining koa_file as string because calc_strehl_on_sky does not
# iterate over Path object correctly in parsing
koa_file = ["/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits", 
            "/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.04422.fits"]
koa_strehls, koa_fwhms, koa_rmswfes = mu.calc_strehl_on_sky(koa_file, "temp.txt")

/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits
Filter = Br_gamma | Scale (arcsec/px) = 0.009952 | Wavelength (microns) = 2.168500 
dl_peak_flux_ratio: 0.03535875
N2.20160803.03304.fits 1 3.735358291226458 505.9385899694424 519.0859101880255 21.0
N2.20160803.03304.fits 2 3.735358291226458 505.9578013838125 519.1196232459902 23.1
N2.20160803.03304.fits 3 16.521235634711235 523.3383957280662 534.9821100148872 27.3
peak flux ratio =  0.021418035766983742  dl peak flux ratio =  0.03535875
/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits   0.606   244.4  164.42  523.3  535.0

N2.20160803.03304.fits           0.606   244.4  164.42  57603.0382

N2.20160803.04422.fits 1 5.9506093888663845 513.3300039352406 515.5447255577776 21.0
peak flux ratio =  0.020743654427437454  dl peak flux ratio =  0.03535875
/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.04422.fits   0.587   252.0   59.22  513.3  515.5

N2.20160803.04422.fits           0.587  

In [19]:
# Simulation seeds
seeds = np.array([1, 1000, 5000, 10000])
# Simulation types/modes
simtypes = ['piston', 'psd+ncpa-unseen', 'psd+ncpa-seen']
baseroot = Path("/Users/bdigia/work/ao/keck/maos/keck/my_base/")
# Run simulation for each KOA image in koa_file array
for koa in koa_file:
    print(koa)
    with fits.open(koa) as koa_fits:
        hdu = koa_fits[0]
        hdr = hdu.header
        # Zenith angle
        angle = np.degrees(np.arccos(1.0/float(hdr['AIRMASS'])))
        # Calculate atm parameters
        koa_path = Path(koa)
        fried, turbpro, windspds, winddrcts, _, _, _, _, _, _ = mu.estimate_on_sky_conditions(koa, koa_path.parent.as_posix() + "/", verbose=True)

        # STRAP WFS integration time (milli-sec)
        hdr_strap_int_time = float(hdr['STINTTIM'])
        dtrat_strap = (1.0/472.0) / (hdr_strap_int_time / 1000.0)
        # Frame rate for WFS cam (Hz) <-- from FITS header of file above
        hdr_shwfs_frame_rate = float(hdr['WSFRRT'])
        hdr_shwfs_int_time = (1.0/hdr_shwfs_frame_rate)
        dtrat_shwfs = (1.0/472.0) / hdr_shwfs_int_time

        for seed in seeds:
            for mode in simtypes:
                if mode == 'piston':
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits"]
                    # Fetch name of current input PSD FITS file in MAOS config file keck_sim.conf
                    psd_file = ''
                elif mode == 'psd+ncpa-seen':
                    mode = 'surf_wfs1'
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=1; SURFEVL=1; seed=10;'"]
                    psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
                elif mode == 'psd+ncpa-unseen':
                    mode = 'surf_wfs0'
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=0; SURFEVL=1; seed=10;'"]
                    psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
                else:
                    raise ValueError(f"Invalid MAOS simulation type '{type}'. Valid types are currently: 'piston', 'psd+ncpa-seen', 'psd+ncpa-unseen'. See help() for further info")
        
                # Must be in MAOS simulation directory to run successfully
                if os.getcwd() != baseroot.as_posix():
                    print("Moving current working directory to MAOS simulation directory...\n")
                    os.chdir(baseroot)

                maos_cmd = f"maos -o A_keck_scao_lgs_koa_{mode}_comp_{koa[60:65]}_seed{seed}_epoch{koa[51:59]} -c A_keck_scao_lgs.conf powfs.dtrat={[dtrat_strap dtrat_shwfs 7080]} sim.seeds={seed} sim.zadeg={angle} sim.wspsd={psd_file} atm.r0z={fried} atm.wt={turbpro} atm.ws={windspds} atm.wddeg={winddrcts} surf={surf_cmd} -O"
                os.system(maos_cmd)

/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits
NOTE: Results for MAOS configuration files marked with ***

20160803.dimm.dat exists in directory /Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/, not downloading.
20160803.masspro.dat exists in directory /Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/, not downloading.
cfht-wx.2016.dat exists in directory /Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/, not downloading.
phto.20160803.dat exists in directory /Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/, not downloading.

Date of observation is  2016-08-03 (UT)
Exposure time is	00:55:05.386 to 00:55:19.945 (UT)

Could not locate DIMM data close to  0.9180555555555555
Could not locate MASS data close to  0.9180555555555555
Could not locate DIMM data close to  0.9219444444444445
Could not locate MASS data close to  0.9219444444444445
Closest DIMM data to beginning of exposure:	0.8500	at 2016:8:3:8:12:2
Closest DIMM data to end of exposure:		0.8500	at 2016:8:3:8:

Warning(sock.c:517): connect locally (/Users/bdigia/.aos/tmp/spray/scheduler) failed: No such file or directory. 
Warning(readcfg.c:406): sim.apfsm has been renamed to powfs.apfsm.
Warning(readcfg.c:406): sim.epfsm has been renamed to powfs.epfsm.
Warning(readcfg.c:406): sim.alfsm has been renamed to powfs.alfsm.
Warning(readcfg.c:406): sim.commonfsm has been renamed to powfs.commonfsm.
Warning(readcfg.c:406): sim.idealfsm has been renamed to powfs.idealfsm.
Warning(readcfg.c:406): sim.zetafsm has been renamed to powfs.zetafsm.
Warning(readcfg.c:406): sim.f0fsm has been renamed to powfs.f0fsm.
Warning(readcfg.c:406): surf has been renamed to ncpa.surf.
Warning(readcfg.c:406): tsurf has been renamed to ncpa.tsurf.
Warning(readcfg.c:406): surf has been renamed to ncpa.surf.
Warning(sock.c:517): connect locally (/Users/bdigia/.aos/tmp/spray/scheduler) failed: No such file or directory. 
Warning(accphi.c:478): 7318 points not covered by input screen


Tomography grid is not square:
    layer 0: xloc grid is  64 x  64, sampling is 0.281 m,  1489 points
    layer 1: xloc grid is  64 x  64, sampling is 0.280 m,  1521 points
    layer 2: xloc grid is  64 x  64, sampling is 0.278 m,  1521 points
    layer 3: xloc grid is  64 x  64, sampling is 0.275 m,  1549 points
    layer 4: xloc grid is  64 x  64, sampling is 0.268 m,  1633 points
    layer 5: xloc grid is  64 x  64, sampling is 0.255 m,  1781 points
    layer 6: xloc grid is  64 x  64, sampling is 0.229 m,  2169 points
Setting up surface OPD

Surface 0:
Loading surface OPD from Keck_ncpa_rmswfe130nm.fits
Does not contain SURFEVL. Assume it covers all evaluation directions.
Does not contain SURFWFS, Assume it covers all WFS.

Setting up powfs 0 PO WFS

OTF dimension is 112x112 (calculated)
Uplink FWHM (illt 0, iwvl 0) is 0.719189"
Downlink FWHM (illt 0, iwvl 0) is 0.877378"
Generating matched filter for powfs 0
sa index   location       noise equivalent angle
     118: (  1.2,  -0.5)